In [3]:
"""Spatial-temporal Fay-Herriot (STFH) for East Java subdistrict panel 2018-2025.

Statistical model based on the supplied R sae::eblupSTFH implementation:
 y_dt = x_dt beta + u_d + v_dt + e_dt
 u ~ N(0, sigma21 * [(I-rho1 W)'(I-rho1 W)]^{-1})
 v_d,t = rho2*v_d,t-1 + innovation, innovation variance sigma22
 e_dt ~ N(0, vardir_dt), independently.

Variance components estimated via *numerical REML* (scipy L-BFGS-B),
NOT R's Fisher-scoring implementation. The same covariance and EBLUP
formula are used, but numerical estimates may differ from R.

Avoids the 5,288 x 5,288 dense covariance matrix using the Woodbury identity.
Requires: pandas numpy scipy geopandas shapely patsy matplotlib pyproj.

Run from the working directory containing the 'data' folder:
    python stfh_python_jatim_2018_2025.py
"""

from __future__ import annotations

import json
import warnings
from pathlib import Path

import geopandas as gpd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import patsy
from scipy import sparse
from scipy.linalg import solve_triangular
from scipy.optimize import minimize
from scipy.sparse.linalg import splu
from scipy.spatial import cKDTree

# ======================== 1. KONFIGURASI ========================
DATA_CSV = Path("data/data_panel_sae_2018_2025.csv")
DATA_SHP = Path("data/shp/kecamatan_jatim.shp")
OUTPUT = Path("data/output_python_stfh")
YEARS = list(range(2018, 2026))
FORMULA = "pengeluaran_mean ~ CO + LST + NDVI + NDBI + NTL + listrik + pendidikan + ekonomi"
AREA_KEY_PANEL = "kode_kecamatan_kemendagri"
AREA_KEY_SHP = "kode_kec"
BPS_KEY = "kode_kecamatan_bps"
TIME_KEY = "tahun"
VARDIR_KEY = "VAR...8"
MODEL = "ST"  # "ST" (spatial + temporal) or "S" (spatial only)
MAXITER = 200
REQUIRE_ALL_PANEL_CODES_IN_SHP = True
K_NEIGHBORS_FOR_ISLANDS = 1
N_BOOTSTRAP = 0  # set e.g. 30; each replicate refits model (computationally costly)
SEED = 2026

COVARS = ["CO", "LST", "NDVI", "NDBI", "NTL", "listrik", "pendidikan", "ekonomi"]
VARS = ["pengeluaran_mean", *COVARS, VARDIR_KEY]


# ======================== 2. READ + VALIDATE ========================
def tidy_code(series: pd.Series) -> pd.Series:
    """Preserve codes as text; normalize DBF-derived decimals only."""
    s = series.astype("string").str.strip()
    s = s.str.replace(r"^(\d+)\.0+$", r"\1", regex=True)
    return s.replace({"": pd.NA, "nan": pd.NA, "None": pd.NA})


def read_panel() -> pd.DataFrame:
    if not DATA_CSV.exists():
        raise FileNotFoundError(f"CSV tidak ditemukan: {DATA_CSV.resolve()}")
    panel = pd.read_csv(DATA_CSV, dtype={AREA_KEY_PANEL: "string", BPS_KEY: "string"}, low_memory=False)
    panel.columns = panel.columns.str.strip()
    unnamed = [x for x in panel if not x or x.startswith("Unnamed:")]
    if unnamed:
        print("Kolom tanpa nama/indeks CSV dihapus:", unnamed)
        panel = panel.drop(columns=unnamed)
    required = [TIME_KEY, AREA_KEY_PANEL, BPS_KEY, *VARS]
    missing = sorted(set(required) - set(panel.columns))
    if missing:
        raise ValueError(f"Kolom CSV tidak ditemukan: {missing}. Kolom tersedia: {panel.columns.tolist()}")
    panel[AREA_KEY_PANEL] = tidy_code(panel[AREA_KEY_PANEL])
    panel[BPS_KEY] = tidy_code(panel[BPS_KEY])
    panel[TIME_KEY] = pd.to_numeric(panel[TIME_KEY], errors="coerce")
    if panel[TIME_KEY].isna().any():
        raise ValueError("Tahun mengandung nilai tidak numerik/NA")
    panel[TIME_KEY] = panel[TIME_KEY].astype(int)
    panel = panel[panel[TIME_KEY].isin(YEARS)].copy()
    for key in VARS:
        panel[key] = pd.to_numeric(panel[key], errors="coerce")
    if panel[AREA_KEY_PANEL].isna().any():
        raise ValueError("Kode Kemendagri panel mengandung NA/kosong")
    if panel.duplicated([AREA_KEY_PANEL, TIME_KEY]).any():
        dup = panel.loc[panel.duplicated([AREA_KEY_PANEL, TIME_KEY], keep=False), [AREA_KEY_PANEL, TIME_KEY]]
        dup.to_csv(OUTPUT / "duplikasi_area_tahun.csv", index=False)
        raise ValueError("Ada duplikasi kecamatan-tahun; cek duplikasi_area_tahun.csv")
    # One BPS identifier per Kemendagri identifier (unless BPS entirely missing)
    mismatch = panel.groupby(AREA_KEY_PANEL)[BPS_KEY].nunique(dropna=True)
    if (mismatch > 1).any():
        warnings.warn("Sebagian kecamatan mempunyai lebih dari satu kode BPS; periksa perubahan administrasi")
    return panel


def read_filter_shp(panel: pd.DataFrame) -> gpd.GeoDataFrame:
    if not DATA_SHP.exists():
        raise FileNotFoundError(f"Shapefile tidak ditemukan: {DATA_SHP.resolve()}")
    shp = gpd.read_file(DATA_SHP)
    if AREA_KEY_SHP not in shp.columns:
        raise ValueError(f"Kolom '{AREA_KEY_SHP}' tidak ada di SHP: {shp.columns.tolist()}")
    if shp.crs is None:
        raise ValueError("SHP tidak memiliki CRS. Tentukan CRS yang benar sebelum menghitung jarak KNN")
    shp[AREA_KEY_SHP] = tidy_code(shp[AREA_KEY_SHP])
    shp = shp[shp[AREA_KEY_SHP].notna()].copy()
    if shp.geometry.isna().any() or shp.geometry.is_empty.any():
        shp = shp[shp.geometry.notna() & ~shp.geometry.is_empty].copy()
    # Make valid before dissolving duplicate codes
    shp.geometry = shp.geometry.make_valid()
    if shp[AREA_KEY_SHP].duplicated().any():
        print("Fitur SHP berkode sama digabung (dissolve)")
        shp = shp[[AREA_KEY_SHP, "geometry"]].dissolve(by=AREA_KEY_SHP, as_index=False)
    shp = shp[[AREA_KEY_SHP, "geometry"]].copy()
    codes = set(panel[AREA_KEY_PANEL].unique().tolist())
    shp = shp[shp[AREA_KEY_SHP].isin(codes)].sort_values(AREA_KEY_SHP).reset_index(drop=True)
    missing_codes = sorted(codes - set(shp[AREA_KEY_SHP]))
    pd.DataFrame({AREA_KEY_PANEL: missing_codes}).to_csv(OUTPUT / "kode_panel_tanpa_shp.csv", index=False)
    if missing_codes:
        msg = f"{len(missing_codes)} kode panel tidak ada di SHP (cek output kode_panel_tanpa_shp.csv)."
        if REQUIRE_ALL_PANEL_CODES_IN_SHP:
            raise ValueError(msg + " Perbaiki kode/join sebelum estimasi.")
        warnings.warn(msg + " Data panel yang tidak cocok akan dikeluarkan.")
    if len(shp) < 2:
        raise ValueError("Hanya tersisa <2 kecamatan pada SHP yang cocok")
    return shp


def align_panel(panel: pd.DataFrame, shp: gpd.GeoDataFrame) -> tuple[pd.DataFrame, int, int]:
    code_order = shp[AREA_KEY_SHP].astype(str).tolist()
    panel = panel[panel[AREA_KEY_PANEL].isin(code_order)].copy()
    expected = pd.MultiIndex.from_product([code_order, YEARS], names=[AREA_KEY_PANEL, TIME_KEY])
    key_df = pd.MultiIndex.from_frame(panel[[AREA_KEY_PANEL, TIME_KEY]])
    missing = expected.difference(key_df)
    if len(missing):
        pd.DataFrame(missing.tolist(), columns=[AREA_KEY_PANEL, TIME_KEY]).to_csv(
            OUTPUT / "panel_tahun_tidak_lengkap.csv", index=False
        )
        raise ValueError(f"Panel tidak lengkap: {len(missing)} pasangan area-tahun hilang; cek CSV audit")
    panel[AREA_KEY_PANEL] = pd.Categorical(panel[AREA_KEY_PANEL], categories=code_order, ordered=True)
    panel = panel.sort_values([AREA_KEY_PANEL, TIME_KEY]).reset_index(drop=True)
    panel[AREA_KEY_PANEL] = panel[AREA_KEY_PANEL].astype(str)
    if not np.array_equal(panel[TIME_KEY].to_numpy().reshape(len(shp), len(YEARS)), np.tile(YEARS, (len(shp), 1))):
        raise ValueError("Urutan tahun bukan 2018-2025 pada setiap kecamatan")
    bad = panel[VARS].isna().any(axis=1)
    if bad.any():
        panel.loc[bad, [AREA_KEY_PANEL, TIME_KEY, *VARS]].to_csv(OUTPUT / "input_nilai_hilang.csv", index=False)
        raise ValueError(f"Ditemukan {bad.sum()} baris dengan NA; cek input_nilai_hilang.csv")
    if not np.isfinite(panel[VARS].to_numpy(dtype=float)).all():
        raise ValueError("Variabel model mengandung inf/-inf")
    if (panel[VARDIR_KEY] <= 0).any():
        panel.loc[panel[VARDIR_KEY] <= 0, [AREA_KEY_PANEL, TIME_KEY, VARDIR_KEY]].to_csv(
            OUTPUT / "vardir_tidak_positif.csv", index=False
        )
        raise ValueError("Sampling variance harus positif; cek vardir_tidak_positif.csv")
    assert panel.shape[0] == len(shp) * len(YEARS)
    return panel, len(shp), len(YEARS)


# ======================== 3. QUEEN + KNN ISLANDS ========================
def build_queen_knn(shp: gpd.GeoDataFrame) -> tuple[sparse.csr_matrix, pd.DataFrame]:
    D = len(shp)
    # touches = adjacent via shared boundary point/edge (Queen geometry criterion)
    adjacent = [set() for _ in range(D)]
    for i, geom in enumerate(shp.geometry):
        indices = shp.sindex.query(geom, predicate="touches")
        for j in indices:
            j = int(j)
            if i != j:
                adjacent[i].add(j)
                adjacent[j].add(i)
    islands = [i for i, neigh in enumerate(adjacent) if not neigh]
    print(f"Island Queen sebelum KNN: {len(islands)}")
    audit = []
    if islands:
        # project into local meter-based CRS, never run KNN with long/lat degrees
        metric_crs = shp.estimate_utm_crs()
        if metric_crs is None:
            raise ValueError("Tidak bisa menentukan CRS metrik untuk KNN")
        metric = shp.to_crs(metric_crs)
        centers = metric.geometry.representative_point()
        coords = np.column_stack((centers.x.to_numpy(), centers.y.to_numpy()))
        tree = cKDTree(coords)
        for i in islands:
            # Always skip itself, even for duplicate representative coordinates
            _, ids = tree.query(coords[i], k=D)
            choices = [int(j) for j in np.atleast_1d(ids) if int(j) != i]
            for j in choices[:K_NEIGHBORS_FOR_ISLANDS]:
                adjacent[i].add(j)
                adjacent[j].add(i)
                dist_m = float(np.linalg.norm(coords[i] - coords[j]))
                audit.append({
                    "kode_island": shp.at[i, AREA_KEY_SHP],
                    "kode_tetangga_knn": shp.at[j, AREA_KEY_SHP],
                    "jarak_titik_meter": dist_m,
                })
        print(f"Island sesudah KNN: {sum(len(nei) == 0 for nei in adjacent)}")
    rows, cols, vals = [], [], []
    for i, neighbors in enumerate(adjacent):
        if not neighbors:
            raise ValueError(f"Area {shp.at[i, AREA_KEY_SHP]} masih tidak memiliki tetangga")
        for j in sorted(neighbors):
            rows.append(i)
            cols.append(j)
            vals.append(1.0 / len(neighbors))
    W = sparse.csr_matrix((vals, (rows, cols)), shape=(D, D))
    if not np.allclose(np.asarray(W.sum(axis=1)).ravel(), 1.0, atol=1e-10):
        raise ValueError("Matriks W tidak row-standardized")
    if np.any(W.diagonal() != 0):
        raise ValueError("Diagonal matriks W tidak nol")
    return W, pd.DataFrame(audit, columns=["kode_island", "kode_tetangga_knn", "jarak_titik_meter"])


# ======================== 4. STFH MATRIX IMPLEMENTATION ========================
class STFH:
    """Restricted maximum likelihood STFH, covariance matching sae::eblupSTFH."""

    def __init__(self, panel: pd.DataFrame, W: sparse.spmatrix, D: int, T: int,
                 model: str = "ST", formula: str = FORMULA):
        if model not in ("S", "ST"):
            raise ValueError("model harus 'S' atau 'ST'")
        self.model = model
        self.formula = formula
        self.D, self.T, self.N = D, T, D * T
        self.W = W.tocsc()
        self.eye = sparse.eye(D, format="csc")
        # Patsy uses R-like formula syntax, preserving intercept
        yy, XX = patsy.dmatrices(formula, panel, return_type="dataframe", NA_action="raise")
        self.names = XX.columns.to_list()
        if "Intercept" not in self.names:
            raise ValueError("Skrip ini memerlukan intercept di formula")
        self.X_raw = XX.to_numpy(dtype=float)
        self.y_raw = yy.to_numpy(dtype=float).ravel()
        self.var_raw = panel[VARDIR_KEY].to_numpy(dtype=float)
        if len(self.y_raw) != self.N:
            raise ValueError("Patsy memodifikasi jumlah baris. Cek missing data")
        if np.linalg.matrix_rank(self.X_raw) < self.X_raw.shape[1]:
            raise ValueError("Variabel penyerta memiliki multikolinearitas sempurna")
        self.y_center = float(self.y_raw.mean())
        self.y_scale = max(float(np.std(self.y_raw)), float(np.sqrt(np.median(self.var_raw))), 1e-8)
        self.y = (self.y_raw - self.y_center) / self.y_scale
        self.var = self.var_raw / self.y_scale**2
        self.x_center = np.zeros(self.X_raw.shape[1])
        self.x_scale = np.ones(self.X_raw.shape[1])
        for j, name in enumerate(self.names):
            if name == "Intercept":
                continue
            self.x_center[j] = self.X_raw[:, j].mean()
            self.x_scale[j] = self.X_raw[:, j].std()
            if self.x_scale[j] <= 0:
                raise ValueError(f"Variabel {name} konstan dan tidak dapat digunakan")
        self.X = (self.X_raw - self.x_center) / self.x_scale
        self.p = self.X.shape[1]
        self.R = np.column_stack([self.y, self.X]).reshape(D, T, self.p + 1)
        self.ysq = self.y.reshape(D, T)
        self.vmat = self.var.reshape(D, T)
        self.tm = np.abs(np.subtract.outer(np.arange(T), np.arange(T)))
        self.result = None

    def _evaluate(self, theta: np.ndarray, details=False):
        # theta = [log sigma21, rho1, log sigma22, rho2] (rho2 absent in S)
        s1, rho1, s2 = np.exp(theta[0]), theta[1], np.exp(theta[2])
        rho2 = theta[3] if self.model == "ST" else 0.0
        try:
            if self.model == "ST":
                temporal = (rho2 ** self.tm) / (1.0 - rho2**2)
            else:
                temporal = np.eye(self.T)
            A = np.tile(s2 * temporal, (self.D, 1, 1))
            idx = np.arange(self.T)
            A[:, idx, idx] += self.vmat
            # A_d = temporal covariance + sampling variance
            sign_A, logdet_A = np.linalg.slogdet(A)
            if np.any(sign_A <= 0):
                return np.inf
            # A^{-1}[y,X] and A^{-1}1 as D stacks of 8x8 systems
            ainv_R = np.linalg.solve(A, self.R)
            ainv_one = np.linalg.solve(A, np.ones((self.D, self.T, 1)))
            c = ainv_R.sum(axis=1)  # Z' A^-1 [y,X]
            s = ainv_one.sum(axis=1).ravel()  # Z' A^-1 Z diagonal
            B = self.eye - rho1 * self.W
            Q = (B.T @ B).tocsc() * (1.0 / s1)  # precision of spatial u1
            H = (Q + sparse.diags(s, format="csc")).tocsc()
            lu_B = splu(B)
            lu_H = splu(H)
            logdet_B = np.log(np.abs(lu_B.U.diagonal())).sum()
            logdet_H = np.log(np.abs(lu_H.U.diagonal())).sum()
            # Woodbury determinant: |V| = |A| * |H| / |Q|
            logdet_Q = 2.0 * logdet_B - self.D * np.log(s1)
            logdet_V = float(logdet_A.sum() + logdet_H - logdet_Q)
            z = lu_H.solve(c)
            invV_R = ainv_R - ainv_one * z[:, None, :]
            cross = self.R.reshape(self.N, -1).T @ invV_R.reshape(self.N, -1)
            cross = (cross + cross.T) / 2
            XtVX = cross[1:, 1:]
            sign_X, logdet_X = np.linalg.slogdet(XtVX)
            if sign_X <= 0:
                return np.inf
            beta = np.linalg.solve(XtVX, cross[1:, 0])
            quad = max(0.0, float(cross[0, 0] - cross[0, 1:] @ beta))
            nll_reml = 0.5 * ((self.N - self.p) * np.log(2 * np.pi) +
                              logdet_V + logdet_X + quad)
            if not details:
                return float(nll_reml) if np.isfinite(nll_reml) else np.inf
            # Posterior spatial E[u|y] = H^{-1} Z' A^{-1} residual
            u1_hat = z[:, 0] - z[:, 1:] @ beta
            # V^{-1} residual
            vinvr = invV_R[:, :, 0] - np.einsum("dtp,p->dt", invV_R[:, :, 1:], beta)
            u2_hat = s2 * (np.einsum("ts,ds->dt", temporal, vinvr) if self.model == "ST" else vinvr)
            fit_scaled = (self.X @ beta).reshape(self.D, self.T) + u1_hat[:, None] + u2_hat
            fit_raw = self.y_center + self.y_scale * fit_scaled.reshape(-1)
            beta_raw = self.y_scale * beta / self.x_scale
            int_idx = self.names.index("Intercept")
            beta_raw[int_idx] += self.y_center - float(self.x_center @ beta_raw)
            ml_ll = -0.5 * (self.N * np.log(2 * np.pi) + logdet_V + quad)
            ml_ll -= self.N * np.log(self.y_scale)
            return {
                "reml_nll_scaled": float(nll_reml),
                "loglik_ml_at_reml": float(ml_ll),
                "beta": dict(zip(self.names, beta_raw.tolist())),
                "theta": {
                    "sigma21": float(s1 * self.y_scale**2),
                    "rho1": float(rho1),
                    "sigma22": float(s2 * self.y_scale**2),
                    "rho2": float(rho2),
                },
                "fitted": fit_raw,
                "spatial_effect": u1_hat * self.y_scale,
                "temporal_effect": u2_hat * self.y_scale,
                "gls_beta_cov_scaled": np.linalg.inv(XtVX),
                "npar": 4 if self.model == "ST" else 3,
            }
        except (ValueError, np.linalg.LinAlgError, RuntimeError, FloatingPointError):
            if details:
                raise
            return np.inf

    def fit(self, maxiter=MAXITER, multistart=True):
        # Variance starts in normalized y units
        initial_var = max(float(np.median(self.var)), 1e-4)
        starts = [(0.25, 0.30), (0.0, 0.0), (0.5, 0.5)] if multistart else [(0.25, 0.30)]
        best = None
        best_converged = None
        for idx, (rho1, rho2) in enumerate(starts, 1):
            x0 = [np.log(initial_var * 0.5), rho1, np.log(initial_var * 0.5)]
            bounds = [(-20.0, 12.0), (-0.98, 0.98), (-20.0, 12.0)]
            if self.model == "ST":
                x0 += [rho2]
                bounds += [(-0.98, 0.98)]
            print(f"REML initial point {idx}/{len(starts)}: rho1={rho1:.2f}, rho2={rho2:.2f}", flush=True)
            opt = minimize(self._evaluate, x0=np.asarray(x0), method="L-BFGS-B",
                           bounds=bounds, options={"maxiter": maxiter, "ftol": 1e-9, "gtol": 1e-5, "maxls": 25})
            print(f"  success={opt.success}, iterations={opt.nit}, REML_nll={opt.fun:.6f}: {opt.message}", flush=True)
            if np.isfinite(opt.fun) and (best is None or opt.fun < best.fun):
                best = opt
            if opt.success and np.isfinite(opt.fun) and (best_converged is None or opt.fun < best_converged.fun):
                best_converged = opt
        if best_converged is not None:
            best = best_converged
        if best is None:
            raise RuntimeError("Optimisasi REML gagal di semua initial point; cek skala data/varians/W")
        if not best.success:
            warnings.warn("Best REML optimizer belum melaporkan konvergensi: " + str(best.message))
        detail = self._evaluate(best.x, details=True)
        detail["convergence"] = bool(best.success)
        detail["iterations"] = int(best.nit)
        detail["optimizer_message"] = str(best.message)
        detail["AIC_approx"] = float(-2 * detail["loglik_ml_at_reml"] + 2 * (self.p + detail["npar"]))
        detail["BIC_approx"] = float(-2 * detail["loglik_ml_at_reml"] + np.log(self.N) * (self.p + detail["npar"]))
        self.result = detail
        return detail


def bootstrap_mse(model: STFH, fitted: dict, B: int, seed: int = SEED) -> np.ndarray:
    """Parametric bootstrap MSE: simulate full latent truth and refit variance components.

    The fitted beta and variance components from original data define the DGP.
    Returns the average squared difference between reestimated EBLUP and true theta.
    This method is computationally expensive; inspect convergence on replications.
    """
    if B <= 0:
        raise ValueError("B harus positif")
    rng = np.random.default_rng(seed)
    theta = fitted["theta"]
    s1, r1 = theta["sigma21"], theta["rho1"]
    s2, r2 = theta["sigma22"], theta["rho2"]
    Bmat = model.eye - r1 * model.W
    lu = splu(Bmat.tocsc())
    beta_vec = np.array([fitted["beta"][name] for name in model.names])
    mu = model.X_raw @ beta_vec
    mse_sum = np.zeros(model.N)
    good = 0
    for b in range(B):
        u = lu.solve(np.sqrt(s1) * rng.standard_normal(model.D))
        temporal = np.zeros((model.D, model.T))
        temporal[:, 0] = rng.normal(scale=np.sqrt(s2 / (1.0 - r2**2)), size=model.D)
        for t in range(1, model.T):
            temporal[:, t] = r2 * temporal[:, t-1] + rng.normal(scale=np.sqrt(s2), size=model.D)
        true_theta = mu + np.repeat(u, model.T) + temporal.reshape(-1)
        y_star = true_theta + rng.normal(scale=np.sqrt(model.var_raw), size=model.N)
        # substitute y in a copy; only response changes, all other inputs held fixed
        simulated = model.source_panel.copy()
        simulated["pengeluaran_mean"] = y_star
        mod_b = STFH(simulated, model.W, model.D, model.T, model.model, model.formula)
        res_b = mod_b.fit(maxiter=MAXITER, multistart=False)
        if res_b["convergence"]:
            mse_sum += (res_b["fitted"] - true_theta)**2
            good += 1
        print(f"  bootstrap {b+1}/{B}, converged={good}", flush=True)
    if good < B:
        warnings.warn(f"Hanya {good}/{B} bootstrap konvergen: MSE berdasarkan replikasi sukses saja")
    if good == 0:
        raise RuntimeError("Tidak ada bootstrap yang konvergen")
    return mse_sum / good


# ======================== 5. RUN + EXPORT ========================
def main():
    OUTPUT.mkdir(parents=True, exist_ok=True)
    print("[1] Membaca dan memvalidasi data panel", flush=True)
    panel = read_panel()
    print("[2] Membaca dan memfilter SHP ke area panel", flush=True)
    shp = read_filter_shp(panel)
    print("[3] Melengkapi dan mengurutkan panel sesuai SHP", flush=True)
    panel, D, T = align_panel(panel, shp)
    print(f"D = {D} kecamatan, T = {T} tahun, N = {len(panel)} baris", flush=True)
    print("[4] Membuat matriks W Queen + KNN hanya untuk island", flush=True)
    W, audit_knn = build_queen_knn(shp)
    audit_knn.to_csv(OUTPUT / "audit_island_knn.csv", index=False)
    codes = shp[AREA_KEY_SHP].tolist()
    pd.DataFrame(W.toarray(), index=codes, columns=codes).to_csv(OUTPUT / "W_Queen_KNN.csv")
    print("[5] Estimasi REML model STFH", flush=True)
    fit_model = STFH(panel, W, D, T, MODEL, FORMULA)
    fit_model.source_panel = panel.copy()
    fitted = fit_model.fit()
    print("Konvergensi:", fitted["convergence"], "|", fitted["optimizer_message"], flush=True)
    print("Estimasi parameter:", fitted["theta"], flush=True)
    print("[6] Ekspor EBLUP kecamatan-tahun", flush=True)
    hasil = panel.copy()
    hasil["EBLUP_STFH"] = fitted["fitted"]
    hasil["selisih_eblup_direct"] = hasil["EBLUP_STFH"] - hasil["pengeluaran_mean"]
    hasil["efek_spasial"] = np.repeat(fitted["spatial_effect"], T)
    hasil["efek_temporal"] = fitted["temporal_effect"].reshape(-1)
    hasil.to_csv(OUTPUT / "hasil_eblup_stfh_2018_2025.csv", index=False)
    for tahun, frame in hasil.groupby(TIME_KEY, sort=True):
        frame.to_csv(OUTPUT / f"hasil_eblup_stfh_{tahun}.csv", index=False)
    pd.DataFrame({"koefisien": fitted["beta"]}).to_csv(OUTPUT / "koefisien_regresi.csv")
    meta = {key: val for key, val in fitted.items() if key not in
            ("fitted", "spatial_effect", "temporal_effect", "gls_beta_cov_scaled")}
    meta.update({"model": MODEL, "formula": FORMULA, "D": D, "T": T, "N": len(panel),
                 "estimator": "REML + L-BFGS-B; not R sae Fisher-scoring",
                 "spatial_neighbors": "Queen, KNN k=1 only for islands, symmetric then row-standardized"})
    (OUTPUT / "diagnostik_model.json").write_text(json.dumps(meta, indent=2, ensure_ascii=False), encoding="utf-8")
    pd.DataFrame([fitted["theta"]]).to_csv(OUTPUT / "parameter_stfh.csv", index=False)
    if N_BOOTSTRAP > 0:
        print(f"[7] Bootstrap parametrik, B={N_BOOTSTRAP}", flush=True)
        mse = bootstrap_mse(fit_model, fitted, N_BOOTSTRAP)
        hasil["MSE_STFH_bootstrap"] = mse
        hasil["RMSE_STFH_bootstrap"] = np.sqrt(mse)
        hasil["CV_STFH_bootstrap"] = np.where(
            hasil["EBLUP_STFH"].to_numpy() != 0,
            100 * np.sqrt(mse) / np.abs(hasil["EBLUP_STFH"].to_numpy()), np.nan)
        hasil.to_csv(OUTPUT / "hasil_eblup_stfh_mse_cv.csv", index=False)
    print("[8] Join estimasi ke peta 2025", flush=True)
    attr_2025 = hasil.loc[hasil[TIME_KEY] == 2025].drop(columns="geometry", errors="ignore")
    map_2025 = shp.merge(attr_2025, left_on=AREA_KEY_SHP, right_on=AREA_KEY_PANEL,
                         how="left", validate="one_to_one")
    map_2025.to_file(OUTPUT / "peta_eblup_stfh_2025.gpkg", layer="eblup_2025", driver="GPKG")
    ax = map_2025.plot(column="EBLUP_STFH", cmap="viridis", legend=True,
                       figsize=(13, 9), edgecolor="none", missing_kwds={"color": "lightgray"})
    ax.set_axis_off()
    ax.set_title("EBLUP-STFH Pengeluaran Kecamatan Jawa Timur, 2025")
    plt.tight_layout()
    plt.savefig(OUTPUT / "peta_eblup_stfh_2025.png", dpi=220, bbox_inches="tight")
    plt.close()
    print("Selesai. Hasil disimpan di:", OUTPUT.resolve(), flush=True)
    if not fitted["convergence"]:
        warnings.warn("Estimator belum konvergen; hasil untuk diagnostik saja, jangan digunakan sebagai estimasi final")


if __name__ == "__main__":
    main()


[1] Membaca dan memvalidasi data panel
Kolom tanpa nama/indeks CSV dihapus: ['Unnamed: 0']
[2] Membaca dan memfilter SHP ke area panel
[3] Melengkapi dan mengurutkan panel sesuai SHP
D = 661 kecamatan, T = 8 tahun, N = 5288 baris
[4] Membuat matriks W Queen + KNN hanya untuk island
Island Queen sebelum KNN: 5
Island sesudah KNN: 0
[5] Estimasi REML model STFH
REML initial point 1/3: rho1=0.25, rho2=0.30
  success=True, iterations=16, REML_nll=4143.676405: CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH
REML initial point 2/3: rho1=0.00, rho2=0.00
  success=True, iterations=17, REML_nll=4143.676405: CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH
REML initial point 3/3: rho1=0.50, rho2=0.50
  success=True, iterations=18, REML_nll=4143.676405: CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH
Konvergensi: True | CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH
Estimasi parameter: {'sigma21': 7709580205.509383, 'rho1': 0.9407152768244473, 'sigma22': 45956678969.30627, 'rh

In [4]:
"""Bootstrap MSE STFH untuk data panel kecamatan Jawa Timur, mengadaptasi
alur simulasi `sae::pbmseSTFH` (R) yang diberikan pengguna.

PENTING:
- Mekanisme bootstrap meniru R: fit dasar -> simulate u1, u2, e ->
  y* -> fit ulang -> (EBLUP* - mu*)^2 -> rata-rata B replikasi SUKSES.
- Estimator ulang di Python memakai optimisasi REML L-BFGS-B dari
  `stfh_python_jatim_2018_2025.py`, bukan Fisher-scoring R.
  Karena itu hasil tidak akan selalu sama numeriknya dengan R.
- Memuat hasil fit dasar dan W dari eksekusi skrip STFH Python sebelumnya;
  tidak perlu membangun ulang SHP/W atau menghitung fit dasar.

Contoh:
  python pbmse_stfh_python_jatim.py --b 10 --max-attempts 50
  python pbmse_stfh_python_jatim.py --b 100 --max-attempts 500

Dependensi: kelas STFH dan konfigurasi yang sudah didefinisikan pada sel sebelumnya.
"""
from __future__ import annotations

import argparse
import json
import sys
import traceback
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import sparse
from scipy.optimize import minimize
from scipy.sparse.linalg import splu

# Menggunakan variabel dan kelas langsung dari kernel notebook yang telah dieksekusi sebelumnya
try:
    STFH
except NameError:
    raise NameError("Silakan jalankan sel kode pertama (c4Qu2EYKFXs6) terlebih dahulu untuk mendefinisikan kelas STFH dan variabel lainnya.")

OUTPUT_DEFAULT = Path("data/output_python_stfh")
COVARS = ["CO", "LST", "NDVI", "NDBI", "NTL", "listrik", "pendidikan", "ekonomi"]


def read_inputs(folder: Path):
    """Load saved original fit, W, and area-time panel with strict key checks."""
    required_files = [
        "hasil_eblup_stfh_2018_2025.csv", "W_Queen_KNN.csv",
        "parameter_stfh.csv", "koefisien_regresi.csv", "diagnostik_model.json",
    ]
    missing = [x for x in required_files if not (folder / x).exists()]
    if missing:
        raise FileNotFoundError(
            "File hasil STFH utama tidak ditemukan di " + str(folder) + ": " + ", ".join(missing)
            + ". Jalankan sel pertama (c4Qu2EYKFXs6) utama dahulu."
        )
    meta = json.loads((folder / "diagnostik_model.json").read_text(encoding="utf-8"))
    if not bool(meta.get("convergence", False)):
        raise RuntimeError("Model STFH awal belum konvergen. Bootstrap tidak boleh dijalankan.")
    model_type = meta.get("model", "ST")
    if model_type not in ("ST", "S"):
        raise ValueError(f"Model awal tidak dikenal: {model_type}")
    if " ".join(str(meta.get("formula", "")).split()) != " ".join(FORMULA.split()):
        raise ValueError("Formula pada hasil fit lama berbeda dengan FORMULA saat ini; fit ulang dahulu.")

    mat = pd.read_csv(folder / "W_Queen_KNN.csv", index_col=0, dtype=str)
    mat.index = mat.index.astype(str).str.strip()
    mat.columns = mat.columns.astype(str).str.strip()
    codes = mat.index.to_list()
    if len(codes) != len(set(codes)) or len(mat.columns) != len(set(mat.columns)):
        raise ValueError("Kode kecamatan duplikat pada baris atau kolom W")
    if set(codes) != set(mat.columns):
        raise ValueError("Kode baris dan kolom W tidak sama")
    mat = mat.loc[codes, codes]
    mat = mat.apply(pd.to_numeric, errors="raise")
    w_array = mat.to_numpy(dtype=float)
    D = len(codes)
    if w_array.shape != (D, D) or not np.isfinite(w_array).all():
        raise ValueError("W memiliki ukuran atau nilai tidak valid")
    if np.any(w_array < 0) or not np.allclose(w_array.sum(axis=1), 1.0, atol=1e-8):
        raise ValueError("W harus nonnegatif dan setiap baris berjumlah 1")
    if not np.allclose(np.diag(w_array), 0):
        raise ValueError("Diagonal W harus nol")
    W = sparse.csr_matrix(w_array)

    panel = pd.read_csv(
        folder / "hasil_eblup_stfh_2018_2025.csv",
        dtype={AREA_KEY_PANEL: "string", BPS_KEY: "string"}, low_memory=False,
    )
    required_columns = [AREA_KEY_PANEL, TIME_KEY, "pengeluaran_mean", VARDIR_KEY, *COVARS]
    missing_columns = sorted(set(required_columns) - set(panel.columns))
    if missing_columns:
        raise ValueError(f"Kolom hasil EBLUP awal kurang: {missing_columns}")
    panel[AREA_KEY_PANEL] = tidy_code(panel[AREA_KEY_PANEL])
    panel[TIME_KEY] = pd.to_numeric(panel[TIME_KEY], errors="raise").astype(int)
    if panel.duplicated([AREA_KEY_PANEL, TIME_KEY]).any():
        raise ValueError("Ada duplikasi kecamatan-tahun di hasil STFH awal")
    if set(panel[AREA_KEY_PANEL]) != set(codes):
        raise ValueError("Cakupan kode kecamatan panel dan W berbeda")
    years = sorted(panel[TIME_KEY].unique().tolist())
    T = len(years)
    if years != list(range(2018, 2026)):
        raise ValueError(f"Tahun harus 2018-2025, ditemukan: {years}")
    expected = pd.MultiIndex.from_product([codes, years], names=[AREA_KEY_PANEL, TIME_KEY])
    observed = pd.MultiIndex.from_frame(panel[[AREA_KEY_PANEL, TIME_KEY]])
    if len(panel) != D * T or len(expected.difference(observed)):
        raise ValueError("Panel harus lengkap untuk setiap kecamatan dan tahun")
    panel[AREA_KEY_PANEL] = pd.Categorical(panel[AREA_KEY_PANEL], categories=codes, ordered=True)
    panel = panel.sort_values([AREA_KEY_PANEL, TIME_KEY]).reset_index(drop=True)
    panel[AREA_KEY_PANEL] = panel[AREA_KEY_PANEL].astype(str)
    for key in ["pengeluaran_mean", VARDIR_KEY, *COVARS]:
        panel[key] = pd.to_numeric(panel[key], errors="raise")
    if not np.isfinite(panel[["pengeluaran_mean", VARDIR_KEY, *COVARS]].to_numpy(dtype=float)).all():
        raise ValueError("Input model mengandung NA/inf")
    if np.any(panel[VARDIR_KEY].to_numpy() <= 0):
        raise ValueError("VAR...8 harus positif")
    if int(meta.get("D", -1)) != D or int(meta.get("T", -1)) != T:
        raise ValueError("Metadata D/T berbeda dari file W dan panel; fit dasar mungkin tidak cocok")

    theta_df = pd.read_csv(folder / "parameter_stfh.csv")
    theta = {name: float(theta_df.loc[0, name]) for name in ["sigma21", "rho1", "sigma22", "rho2"]}
    if theta["sigma21"] <= 0 or theta["sigma22"] <= 0:
        raise ValueError("Komponen varians fit awal harus positif untuk bootstrap ini")
    if abs(theta["rho1"]) >= 1 or abs(theta["rho2"]) >= 1:
        raise ValueError("Parameter korelasi tidak berada dalam (-1,1)")

    coef_df = pd.read_csv(folder / "koefisien_regresi.csv", index_col=0)
    if "koefisien" not in coef_df.columns:
        raise ValueError("Kolom koefisien regresi tidak ditemukan")
    beta = coef_df["koefisien"].astype(float).to_dict()
    if not all(np.isfinite(list(beta.values()))):
        raise ValueError("Koefisien fit awal mengandung NA/inf")

    return panel, W, model_type, theta, beta, years


def fit_one_bootstrap(model: STFH, sigma21_start: float, rho1_start: float,
                      sigma22_start: float, rho2_start: float, maxiter: int):
    """Fit a bootstrap replicate with starts matching pbmseSTFH R values.

    Objective uses REML; R uses Fisher scoring. Preserve same starts across
    bootstrap draws in ORIGINAL Y units, transformed here to model's internal
    normalized units. Returns (details, optimizer status), or raises.
    """
    sigma1_scaled = sigma21_start / model.y_scale**2
    sigma2_scaled = sigma22_start / model.y_scale**2
    if sigma1_scaled <= 0 or sigma2_scaled <= 0:
        raise ValueError("Nilai awal varians harus positif")
    x0 = [np.log(sigma1_scaled), rho1_start, np.log(sigma2_scaled)]
    bounds = [(-25.0, 25.0), (-0.98, 0.98), (-25.0, 25.0)]
    if model.model == "ST":
        x0.append(rho2_start)
        bounds.append((-0.98, 0.98))
    opt = minimize(
        model._evaluate, x0=np.asarray(x0, dtype=float), method="L-BFGS-B",
        bounds=bounds,
        options={"maxiter": int(maxiter), "ftol": 1e-9, "gtol": 1e-5, "maxls": 25},
    )
    if not opt.success or not np.isfinite(opt.fun):
        raise RuntimeError(f"REML optimizer gagal: {opt.message}")
    details = model._evaluate(opt.x, details=True)
    vals = np.asarray(details["fitted"], dtype=float)
    if len(vals) != model.N or not np.isfinite(vals).all():
        raise RuntimeError("Fit bootstrap menghasilkan EBLUP tidak valid")
    return details, opt


def bootstrap_pbmse_like(panel: pd.DataFrame, W: sparse.spmatrix,
                         model_type: str, theta: dict, beta: dict,
                         B: int, max_attempts: int, maxiter: int, seed: int,
                         output: Path, verbose: bool = True):
    """Mimic pbmseSTFH bootstrap Monte Carlo MSE, conditional on original fit.

    Exactly B SUCCESSFUL replicate refits enter the denominator, following
    the R `if (!convergence) next` then b++ behavior. A finite attempt cap
    prevents an infinite loop when refits systematically fail.
    """
    if B <= 0 or max_attempts < B:
        raise ValueError("B harus >0 dan max_attempts >= B")
    D = W.shape[0]
    T = panel[TIME_KEY].nunique()
    m0 = STFH(panel, W, D, T, model_type, FORMULA)
    mu = np.zeros(len(panel), dtype=float)
    for j, name in enumerate(m0.names):
        if name not in beta:
            raise ValueError(f"Koefisien {name} tidak ada pada hasil fit awal")
        mu += m0.X_raw[:, j] * beta[name]
    vardir = panel[VARDIR_KEY].to_numpy(dtype=float)
    rng = np.random.default_rng(seed)
    L = (sparse.eye(D, format="csc") - theta["rho1"] * W.tocsc()).tocsc()
    Llu = splu(L)
    s1, s2 = float(theta["sigma21"]), float(theta["sigma22"])
    r2 = float(theta["rho2"]) if model_type == "ST" else 0.0

    # Exactly like R's defaults (in original measurement units), unless caller
    # deliberately changes values in the function call below.
    start1 = 0.5 * float(np.median(vardir))
    start2 = 0.5 * float(np.median(vardir))
    rho1_start, rho2_start = 0.5, 0.5

    sumsquares = np.zeros(D * T, dtype=float)
    accepted = 0
    attempts = 0
    log_rows = []
    while accepted < B and attempts < max_attempts:
        attempts += 1
        # Spatial u1 ~ N(0, sigma21 * [(I - rho1 W)'(I - rho1 W)]^-1)
        z = rng.standard_normal(D)
        u1 = Llu.solve(np.sqrt(s1) * z)              # exact covariance
        u1dt = np.repeat(u1, T)                      # area-major time order

        # Temporal u2: stationary AR(1) per area for ST, iid for S
        innovations = rng.standard_normal((D, T)) * np.sqrt(s2)
        v = np.zeros((D, T), dtype=float)
        if model_type == "ST":
            v[:, 0] = innovations[:, 0] / np.sqrt(1.0 - r2**2)
            for t in range(1, T):
                v[:, t] = r2 * v[:, t - 1] + innovations[:, t]
        else:
            v = innovations
        u2dt = v.reshape(-1)

        # e_dt* ~ N(0, vardir_dt) and true latent mean mu_dt*
        e = rng.standard_normal(D * T) * np.sqrt(vardir)
        latent_truth = mu + u1dt + u2dt
        y_star = latent_truth + e
        simulated = panel.copy()
        simulated["pengeluaran_mean"] = y_star
        try:
            rep_model = STFH(simulated, W, D, T, model_type, FORMULA)
            fitted_b, opt = fit_one_bootstrap(
                rep_model, sigma21_start=start1, rho1_start=rho1_start,
                sigma22_start=start2, rho2_start=rho2_start, maxiter=maxiter,
            )
            differences = np.asarray(fitted_b["fitted"]) - latent_truth
            if not np.isfinite(differences).all():
                raise ValueError("Selisih bootstrap mengandung NA/inf")
            sumsquares += differences**2
            accepted += 1
            log_rows.append({"attempt": attempts, "accepted_b": accepted,
                             "success": True, "iterations": int(opt.nit),
                             "message": str(opt.message)})
            if verbose:
                print(f"bootstrap diterima: b={accepted}/{B} (percobaan {attempts}, iter={opt.nit})", flush=True)
        except (RuntimeError, ValueError, np.linalg.LinAlgError, FloatingPointError) as exc:
            log_rows.append({"attempt": attempts, "accepted_b": accepted,
                             "success": False, "iterations": None,
                             "message": f"{type(exc).__name__}: {exc}"})
            if verbose:
                print(f"bootstrap gagal: percobaan {attempts}/{max_attempts}: {exc}", flush=True)

    pd.DataFrame(log_rows).to_csv(output / "riwayat_replikasi_pbmse.csv", index=False)
    if accepted < B:
        # R could loop indefinitely here; do NOT divide by accepted and
        # present it as B-replicate pbmse. Save partial diagnostic only.
        np.save(output / "jumlah_kuadrat_parsial.npy", sumsquares)
        raise RuntimeError(
            f"Hanya {accepted} dari B={B} replikasi sukses setelah {attempts} percobaan. "
            "Belum ada MSE final; lihat riwayat_replikasi_pbmse.csv."
        )
    mse = sumsquares / B
    diagnostics = {"B_sukses": accepted, "jumlah_percobaan": attempts,
                   "jumlah_gagal": attempts - accepted,
                   "model": model_type, "seed": seed,
                   "estimator_python": "numerical REML L-BFGS-B (R uses Fisher scoring)",
                   "method": "R pbmseSTFH-compatible parametric simulation + refit; MSE = sum(diff^2)/B",
                   "maxiter_fit_bootstrap": maxiter,
                   "start_sigma21": start1, "start_sigma22": start2,
                   "start_rho1": rho1_start, "start_rho2": rho2_start if model_type == "ST" else None,
                   "n_area": D, "n_tahun": T}
    return mse, diagnostics


def main():
    # Menjalankan fungsi bootstrap langsung di lingkungan notebook dengan default B=10 untuk contoh cepat
    folder = OUTPUT_DEFAULT
    folder.mkdir(parents=True, exist_ok=True)
    b_replications = 10
    max_attempts = 5 * b_replications

    panel, W, model_type, theta, beta, years = read_inputs(folder)
    print(f"D={W.shape[0]}, T={len(years)}, N={len(panel)}, MODEL={model_type}, B={b_replications}", flush=True)
    print("Parameter awal fit dasar:", theta, flush=True)
    mse, diag = bootstrap_pbmse_like(
        panel, W, model_type, theta, beta, b_replications,
        max_attempts=max_attempts, maxiter=100,
        seed=2026, output=folder,
    )
    hasil = panel.copy()
    hasil["EBLUP_STFH"] = pd.to_numeric(hasil["EBLUP_STFH"], errors="raise")
    hasil["MSE_STFH_pbmse"] = mse
    hasil["RMSE_STFH_pbmse"] = np.sqrt(mse)
    with np.errstate(divide="ignore", invalid="ignore"):
        hasil["CV_STFH_pbmse"] = np.where(
            np.abs(hasil["EBLUP_STFH"].to_numpy()) > 0,
            100.0 * np.sqrt(mse) / np.abs(hasil["EBLUP_STFH"].to_numpy()), np.nan,
        )
        y_direct = hasil["pengeluaran_mean"].to_numpy(dtype=float)
        hasil["CV_direct"] = np.where(
            np.abs(y_direct) > 0,
            100.0 * np.sqrt(hasil[VARDIR_KEY].to_numpy(dtype=float)) / np.abs(y_direct), np.nan,
        )
    hasil.to_csv(folder / "hasil_pbmse_stfh_2018_2025.csv", index=False)
    summary = hasil.groupby(TIME_KEY, sort=True).agg(
        jumlah_kecamatan=(AREA_KEY_PANEL, "nunique"),
        MSE_rata_rata=("MSE_STFH_pbmse", "mean"),
        RMSE_rata_rata=("RMSE_STFH_pbmse", "mean"),
        CV_STFH_rata_rata=("CV_STFH_pbmse", "mean"),
        CV_direct_rata_rata=("CV_direct", "mean"),
    ).reset_index()
    summary.to_csv(folder / "ringkasan_pbmse_per_tahun.csv", index=False)
    (folder / "diagnostik_pbmse.json").write_text(json.dumps(diag, indent=2), encoding="utf-8")
    print("\nBootstrap SELESAI. File:", flush=True)
    print(" -", folder / "hasil_pbmse_stfh_2018_2025.csv")
    print(" -", folder / "ringkasan_pbmse_per_tahun.csv")
    print(" -", folder / "riwayat_replikasi_pbmse.csv")
    print(" -", folder / "diagnostik_pbmse.json")
    print("\nRingkasan:\n", summary.to_string(index=False), flush=True)


if __name__ == "__main__":
    # Deteksi apakah dijalankan sebagai skrip eksternal atau langsung di notebook
    if "ipykernel" in sys.modules:
        main()
    else:
        # Dukungan parameter baris perintah jika dipanggil sebagai berkas .py
        parser = argparse.ArgumentParser(description="Bootstrap MSE STFH ala R pbmseSTFH pada output Python sebelumnya")
        parser.add_argument("--b", type=int, default=100, help="B replikasi bootstrap sukses")
        parser.add_argument("--max-attempts", type=int, default=None, help="Batas percobaan refit; default 5*B")
        parser.add_argument("--maxiter", type=int, default=100, help="Batas iterasi REML per refit, default 100")
        parser.add_argument("--seed", type=int, default=2026)
        parser.add_argument("--folder", type=Path, default=OUTPUT_DEFAULT)
        args = parser.parse_args()
        # Logic running with parsed arguments...

D=661, T=8, N=5288, MODEL=ST, B=10
Parameter awal fit dasar: {'sigma21': 7709580205.509383, 'rho1': 0.9407152768244472, 'sigma22': 45956678969.30627, 'rho2': 0.5922880381145281}
bootstrap diterima: b=1/10 (percobaan 1, iter=14)
bootstrap diterima: b=2/10 (percobaan 2, iter=14)
bootstrap diterima: b=3/10 (percobaan 3, iter=14)
bootstrap diterima: b=4/10 (percobaan 4, iter=13)
bootstrap diterima: b=5/10 (percobaan 5, iter=14)
bootstrap diterima: b=6/10 (percobaan 6, iter=12)
bootstrap diterima: b=7/10 (percobaan 7, iter=14)
bootstrap diterima: b=8/10 (percobaan 8, iter=14)
bootstrap diterima: b=9/10 (percobaan 9, iter=14)
bootstrap diterima: b=10/10 (percobaan 10, iter=12)

Bootstrap SELESAI. File:
 - data/output_python_stfh/hasil_pbmse_stfh_2018_2025.csv
 - data/output_python_stfh/ringkasan_pbmse_per_tahun.csv
 - data/output_python_stfh/riwayat_replikasi_pbmse.csv
 - data/output_python_stfh/diagnostik_pbmse.json

Ringkasan:
  tahun  jumlah_kecamatan  MSE_rata_rata  RMSE_rata_rata  CV_ST

In [5]:
import numpy as np
import pandas as pd
from pathlib import Path

# Membaca kembali hasil bootstrap yang telah disimpan secara lokal
OUTPUT_DIR = Path("data/output_python_stfh")
filepath = OUTPUT_DIR / "hasil_pbmse_stfh_2018_2025.csv"

if not filepath.exists():
    raise FileNotFoundError(f"File {filepath} tidak ditemukan. Pastikan sel 92VYlsi1KglK telah sukses dieksekusi.")

hasil = pd.read_csv(filepath)

# Menghitung RRMSE_STFH
hasil["RRMSE_STFH"] = np.where(
    hasil["EBLUP_STFH"] != 0,
    100 * np.sqrt(hasil["MSE_STFH_pbmse"]) /
    np.abs(hasil["EBLUP_STFH"]),
    np.nan
)

# Menyimpan kembali perubahan ke CSV
hasil.to_csv(filepath, index=False)
print("RRMSE_STFH berhasil dihitung dan disimpan kembali ke hasil_pbmse_stfh_2018_2025.csv")

RRMSE_STFH berhasil dihitung dan disimpan kembali ke hasil_pbmse_stfh_2018_2025.csv


In [6]:
"""Visualisasi estimasi langsung dan EBLUP-STFH Jawa Timur 2018-2025.

Jalankan dari folder proyek (folder yang berisi data/):
    python visualisasi_stfh_jatim.py
    python visualisasi_stfh_jatim.py --hasil data/output_python_stfh/hasil_pbmse_stfh_2018_2025.csv

Membuat peta panel 2 kolom x 4 baris, peta perbandingan 2 kolom x 8 baris,
dan boxplot EBLUP per tahun. Tidak melakukan estimasi ulang.
"""
from __future__ import annotations
import argparse
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
from matplotlib.ticker import FuncFormatter

TAHUN = list(range(2018, 2026))
KEY_PANEL = 'kode_kecamatan_kemendagri'
KEY_SHP = 'kode_kec'
DIRECT = 'pengeluaran_mean'
STFH = 'EBLUP_STFH'


def read_data(csv_path: Path, shp_path: Path):
    if not csv_path.exists():
        raise FileNotFoundError(f'CSV tidak ditemukan: {csv_path}')
    if not shp_path.exists():
        raise FileNotFoundError(f'SHP tidak ditemukan: {shp_path}')
    panel = pd.read_csv(csv_path, dtype={KEY_PANEL: 'string', 'kode_kecamatan_bps': 'string'})
    needed = [KEY_PANEL, 'tahun', DIRECT, STFH]
    missing = [x for x in needed if x not in panel.columns]
    if missing:
        raise ValueError(f'Kolom CSV tidak tersedia: {missing}')
    panel[KEY_PANEL] = panel[KEY_PANEL].astype('string').str.strip()
    panel['tahun'] = pd.to_numeric(panel['tahun'], errors='raise').astype(int)
    panel = panel[panel['tahun'].isin(TAHUN)].copy()
    for col in [DIRECT, STFH]:
        panel[col] = pd.to_numeric(panel[col], errors='coerce')
    if panel.empty:
        raise ValueError('Tidak ada observasi tahun 2018-2025.')
    if panel[[KEY_PANEL, 'tahun']].isna().any().any():
        raise ValueError('Kode kecamatan/tahun memiliki nilai kosong.')
    if panel.duplicated([KEY_PANEL, 'tahun']).any():
        examples = panel.loc[panel.duplicated([KEY_PANEL, 'tahun'], keep=False), [KEY_PANEL, 'tahun']].head(10)
        raise ValueError(f'Duplikasi kecamatan-tahun pada CSV:\n{examples}')
    shp = gpd.read_file(shp_path)
    if KEY_SHP not in shp.columns:
        raise ValueError(f'Kolom {KEY_SHP} tidak ditemukan di SHP.')
    if shp.crs is None:
        raise ValueError('CRS shapefile tidak tersedia. Tetapkan CRS yang benar terlebih dahulu.')
    shp[KEY_SHP] = shp[KEY_SHP].astype('string').str.strip()
    shp = shp[shp[KEY_SHP].isin(panel[KEY_PANEL].unique())].copy()
    if shp.empty:
        raise ValueError('Tidak ada kode kecamatan yang cocok antara CSV dan SHP.')
    shp['geometry'] = shp.geometry.make_valid()
    shp = shp.dissolve(by=KEY_SHP, as_index=False)
    missing_codes = sorted(set(panel[KEY_PANEL].dropna()) - set(shp[KEY_SHP].dropna()))
    if missing_codes:
        raise ValueError(f'{len(missing_codes)} kode panel tidak ditemukan di SHP. Contoh: {missing_codes[:10]}')
    if shp.geometry.is_empty.any() or shp.geometry.isna().any():
        raise ValueError('Ada geometri kosong/tidak tersedia pada SHP.')
    for yr in TAHUN:
        sub = panel[panel.tahun == yr]
        if len(sub) != len(shp) or set(sub[KEY_PANEL]) != set(shp[KEY_SHP]):
            raise ValueError(f'Panel tahun {yr} tidak lengkap terhadap {len(shp)} kecamatan SHP.')
    if panel[[DIRECT, STFH]].isna().any().any() or not np.isfinite(panel[[DIRECT, STFH]].to_numpy()).all():
        raise ValueError('Nilai pengeluaran_mean / EBLUP_STFH mengandung NA atau Inf.')
    return panel, shp


def annual_maps(panel, shp):
    return [shp.merge(panel.loc[panel.tahun == yr, [KEY_PANEL, DIRECT, STFH]],
                      left_on=KEY_SHP, right_on=KEY_PANEL, how='left', validate='one_to_one')
            for yr in TAHUN]


def draw_map(ax, gdf, variable, norm, cmap, title):
    # Mengatur ketebalan batas menjadi 0.3 dan warna batas abu-abu gelap agar lebih jelas terlihat
    gdf.plot(column=variable, ax=ax, cmap=cmap, norm=norm, linewidth=0.3,
             edgecolor='#2c3e50', missing_kwds={'color':'#dddddd'})
    ax.set_title(title, fontsize=11, weight='bold', pad=5)
    ax.set_axis_off()
    ax.set_aspect('equal')


def colorbar(fig, axes, norm, cmap, label):
    sm = ScalarMappable(norm=norm, cmap=cmap)
    sm.set_array([])
    cb = fig.colorbar(sm, ax=np.asarray(axes).ravel().tolist(), orientation='horizontal',
                      fraction=0.025, pad=0.025, shrink=0.70, aspect=35)
    cb.set_label(label, fontsize=10)
    cb.ax.xaxis.set_major_formatter(FuncFormatter(lambda v, pos: f'{v:,.0f}'))


def map_8(geodata, variable, norm, cmap, output, label, dpi):
    fig, axs = plt.subplots(4, 2, figsize=(13, 20), layout='constrained')
    for i, (year, geo) in enumerate(zip(TAHUN, geodata)):
        draw_map(axs.flat[i], geo, variable, norm, cmap, str(year))
    fig.suptitle(f'Sebaran {label} Kecamatan Jawa Timur, 2018–2025', fontsize=17, weight='bold')
    colorbar(fig, axs, norm, cmap, f'{label} (satuan mengikuti CSV)')
    fig.savefig(output, dpi=dpi, bbox_inches='tight')
    plt.close(fig)


def map_compare(geodata, norm, cmap, output, dpi):
    fig, axs = plt.subplots(8, 2, figsize=(15, 32), layout='constrained')
    for i, (year, geo) in enumerate(zip(TAHUN, geodata)):
        draw_map(axs[i, 0], geo, DIRECT, norm, cmap, f'{year} | Estimasi langsung')
        draw_map(axs[i, 1], geo, STFH, norm, cmap, f'{year} | EBLUP-STFH')
    fig.suptitle('Perbandingan Pengeluaran Kecamatan Jawa Timur: Langsung vs STFH', fontsize=17, weight='bold')
    colorbar(fig, axs, norm, cmap, 'Pengeluaran (satuan mengikuti CSV)')
    fig.savefig(output, dpi=dpi, bbox_inches='tight')
    plt.close(fig)


def boxplot(panel, output, dpi, show_outliers=True):
    values = [panel.loc[panel.tahun == year, STFH].to_numpy(dtype=float) for year in TAHUN]
    fig, ax = plt.subplots(figsize=(12, 6.5), layout='constrained')

    # showfliers=show_outliers digunakan untuk menampilkan/menyembunyikan outlier.
    # markersize=6 pada flierprops digunakan untuk memperbesar ukuran titik outlier.
    ax.boxplot(values, tick_labels=[str(y) for y in TAHUN], patch_artist=True,
               showfliers=show_outliers,
               boxprops={'facecolor':'#84b6d9', 'edgecolor':'#34495e'},
               medianprops={'color':'#a43d2c', 'linewidth':2},
               whiskerprops={'color':'#34495e'}, capprops={'color':'#34495e'},
               flierprops={'marker':'.', 'markersize':6, 'alpha':0.5, 'markerfacecolor':'#a43d2c', 'markeredgecolor':'#a43d2c'})

    ax.set_title('Distribusi Pengeluaran per kapita Spasio-Temporal SAE Kecamatan Jawa Timur, 2018–2025', fontsize=14, weight='bold')
    ax.set_xlabel('Tahun')
    ax.set_ylabel('Pengeluaran per kapita')
    ax.yaxis.set_major_formatter(FuncFormatter(lambda v, pos: f'{v:,.0f}'))
    ax.grid(axis='y', alpha=0.25)
    fig.savefig(output, dpi=dpi, bbox_inches='tight')
    plt.close(fig)


def main():
    parser = argparse.ArgumentParser(description='Peta 2018-2025 dan boxplot STFH')
    parser.add_argument('--hasil', type=Path, default=Path('data/output_python_stfh/hasil_eblup_stfh_2018_2025.csv'))
    parser.add_argument('--shp', type=Path, default=Path('data/shp/kecamatan_jatim.shp'))
    parser.add_argument('--output', type=Path, default=Path('data/output_python_stfh/visualisasi'))
    parser.add_argument('--dpi', type=int, default=220)
    parser.add_argument('--cmap', default='YlGn')
    parser.add_argument('--show-outliers', type=bool, default=True, help='Tampilkan outlier pada boxplot atau tidak')

    # Menangani argumen Jupyter notebook
    if 'ipykernel' in sys.modules:
        args = parser.parse_args(args=[])
    else:
        args = parser.parse_args()

    args.output.mkdir(parents=True, exist_ok=True)
    panel, shp = read_data(args.hasil, args.shp)
    geodata = annual_maps(panel, shp)
    vals = panel[[DIRECT, STFH]].to_numpy(dtype=float)
    low, high = float(np.min(vals)), float(np.max(vals))
    if low == high:
        low, high = low - 0.5, high + 0.5
    norm = Normalize(vmin=low, vmax=high)
    map_8(geodata, DIRECT, norm, args.cmap, args.output/'peta_estimasi_langsung_2018_2025.png', 'Estimasi Langsung', args.dpi)
    map_8(geodata, STFH, norm, args.cmap, args.output/'peta_eblup_stfh_2018_2025.png', 'EBLUP-STFH', args.dpi)
    map_compare(geodata, norm, args.cmap, args.output/'peta_perbandingan_langsung_stfh_2018_2025.png', args.dpi)

    # Menggunakan argumen show_outliers (bisa diganti False untuk membuang outlier)
    boxplot(panel, args.output/'boxplot_eblup_stfh_2018_2025.png', args.dpi, show_outliers=args.show_outliers)

    ringkasan = panel.groupby('tahun').agg(n_kecamatan=(STFH,'size'), rata_rata=(STFH,'mean'), median=(STFH,'median'), sd=(STFH,'std'), minimum=(STFH,'min'), maksimum=(STFH,'max')).reset_index()
    ringkasan.to_csv(args.output/'ringkasan_eblup_per_tahun.csv', index=False)
    print(f'Selesai. {len(shp)} kecamatan, {len(TAHUN)} tahun. Output: {args.output.resolve()}')
    for file in sorted(args.output.glob('*')):
        print(' -', file.name)


if __name__ == '__main__':
    main()


Selesai. 661 kecamatan, 8 tahun. Output: /content/data/output_python_stfh/visualisasi
 - boxplot_eblup_stfh_2018_2025.png
 - peta_eblup_stfh_2018_2025.png
 - peta_estimasi_langsung_2018_2025.png
 - peta_perbandingan_langsung_stfh_2018_2025.png
 - ringkasan_eblup_per_tahun.csv


In [7]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import seaborn as sns

# 1. Load Data
OUTPUT_DIR = Path("data/output_python_stfh")
filepath = OUTPUT_DIR / "hasil_pbmse_stfh_2018_2025.csv"
if not filepath.exists():
    raise FileNotFoundError(f"File {filepath} tidak ditemukan.")

hasil = pd.read_csv(filepath)

# 2. Menghitung MSE, RMSE, RRMSE untuk Direct dan STFH
# Direct Estimate (pengeluaran_mean)
hasil["MSE_direct"] = hasil["VAR...8"]
hasil["RMSE_direct"] = np.sqrt(hasil["MSE_direct"])
hasil["RRMSE_direct"] = np.where(
    hasil["pengeluaran_mean"] != 0,
    100 * hasil["RMSE_direct"] / np.abs(hasil["pengeluaran_mean"]),
    np.nan
)

# STFH Estimate (EBLUP_STFH)
hasil["MSE_STFH"] = hasil["MSE_STFH_pbmse"]
hasil["RMSE_STFH"] = np.sqrt(hasil["MSE_STFH"])
hasil["RRMSE_STFH"] = np.where(
    hasil["EBLUP_STFH"] != 0,
    100 * hasil["RMSE_STFH"] / np.abs(hasil["EBLUP_STFH"]),
    np.nan
)

# Simpan kembali ke file CSV
hasil.to_csv(filepath, index=False)
print("✓ Nilai MSE, RMSE, dan RRMSE berhasil digabungkan ke tabel hasil dan disimpan.\n")

# 3. Membuat Summary Perbandingan Tahun ke Tahun
summary_tahun = hasil.groupby("tahun").agg(
    Rata_Rata_MSE_Direct=("MSE_direct", "mean"),
    Rata_Rata_MSE_STFH=("MSE_STFH", "mean"),
    Rata_Rata_RMSE_Direct=("RMSE_direct", "mean"),
    Rata_Rata_RMSE_STFH=("RMSE_STFH", "mean"),
    Rata_Rata_RRMSE_Direct=("RRMSE_direct", "mean"),
    Rata_Rata_RRMSE_STFH=("RRMSE_STFH", "mean"),
    Median_RRMSE_Direct=("RRMSE_direct", "median"),
    Median_RRMSE_STFH=("RRMSE_STFH", "median")
).reset_index()

summary_tahun.to_csv(OUTPUT_DIR / "ringkasan_komparasi_akurasi.csv", index=False)
print("=== RINGKASAN PERBANDINGAN TAHUN KE TAHUN ===")
print(summary_tahun.to_string(index=False))
print("\n")

# 4. Visualisasi Boxplot RRMSE (Direct vs STFH)
plt.figure(figsize=(14, 7))
plot_data = pd.melt(
    hasil,
    id_vars=["tahun"],
    value_vars=["RRMSE_direct", "RRMSE_STFH"],
    var_name="Metode",
    value_name="RRMSE (%)"
)
plot_data["Metode"] = plot_data["Metode"].map({"RRMSE_direct": "Direct (Langsung)", "RRMSE_STFH": "STFH"})

sns.boxplot(data=plot_data, x="tahun", y="RRMSE (%)", hue="Metode", palette="Set2")
plt.title("Perbandingan Distribusi RRMSE (%) per Tahun\nDirect vs STFH Kecamatan Jawa Timur 2018-2025", fontsize=14, fontweight="bold")
plt.xlabel("Tahun", fontsize=12)
plt.ylabel("RRMSE (%)", fontsize=12)
plt.grid(axis="y", linestyle="--", alpha=0.7)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "boxplot_perbandingan_rrmse.png", dpi=300)
plt.close()
print("✓ Boxplot perbandingan RRMSE berhasil disimpan ke: boxplot_perbandingan_rrmse.png")

# 5. Visualisasi Tambahan: Tren Rata-rata RMSE & RRMSE dari Tahun ke Tahun
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Subplot 1: Rata-rata RMSE
axes[0].plot(summary_tahun["tahun"], summary_tahun["Rata_Rata_RMSE_Direct"], marker="o", label="Direct", color="#e74c3c", linewidth=2)
axes[0].plot(summary_tahun["tahun"], summary_tahun["Rata_Rata_RMSE_STFH"], marker="s", label="STFH", color="#2bc0e4", linewidth=2)
axes[0].set_title("Tren Rata-Rata RMSE per Tahun", fontsize=12, fontweight="bold")
axes[0].set_xlabel("Tahun")
axes[0].set_ylabel("Rata-Rata RMSE")
axes[0].legend()
axes[0].grid(True, linestyle=":")

# Subplot 2: Rata-rata RRMSE
axes[1].plot(summary_tahun["tahun"], summary_tahun["Rata_Rata_RRMSE_Direct"], marker="o", label="Direct", color="#e74c3c", linewidth=2)
axes[1].plot(summary_tahun["tahun"], summary_tahun["Rata_Rata_RRMSE_STFH"], marker="s", label="STFH", color="#2bc0e4", linewidth=2)
axes[1].set_title("Tren Rata-Rata RRMSE (%) per Tahun", fontsize=12, fontweight="bold")
axes[1].set_xlabel("Tahun")
axes[1].set_ylabel("Rata-Rata RRMSE (%)")
axes[1].legend()
axes[1].grid(True, linestyle=":")

plt.suptitle("Analisis Performa Akurasi Model: Direct vs STFH", fontsize=15, fontweight="bold")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "tren_akurasi_rmse_rrmse.png", dpi=300)
plt.close()
print("✓ Grafik tren akurasi berhasil disimpan ke: tren_akurasi_rmse_rrmse.png")


✓ Nilai MSE, RMSE, dan RRMSE berhasil digabungkan ke tabel hasil dan disimpan.

=== RINGKASAN PERBANDINGAN TAHUN KE TAHUN ===
 tahun  Rata_Rata_MSE_Direct  Rata_Rata_MSE_STFH  Rata_Rata_RMSE_Direct  Rata_Rata_RMSE_STFH  Rata_Rata_RRMSE_Direct  Rata_Rata_RRMSE_STFH  Median_RRMSE_Direct  Median_RRMSE_STFH
  2018          2.354532e+10        8.721647e+09          103024.570470         77908.851999               10.527358              8.593841             9.580867           8.279837
  2019          5.172260e+10        7.720021e+09          107357.092955         73944.460000               10.330659              8.106422             9.440283           7.807853
  2020          2.391375e+10        8.939291e+09          110968.014194         80902.460257               10.680746              8.426755             9.798144           8.086125
  2021          2.054838e+10        8.626183e+09          104875.119813         78950.334181                8.598814              7.155576             7.60298

In [8]:
import matplotlib.pyplot as plt
import seaborn as sns

# Membuat visualisasi boxplot hanya untuk STFH
plt.figure(figsize=(12, 6.5))

sns.boxplot(
    data=hasil,
    x="tahun",
    y="RRMSE_STFH",
    color="#2bc0e4",
    width=0.5,
    flierprops={'marker': '.', 'markersize': 3, 'alpha': 0.35}
)

plt.title("Distribusi RRMSE (%) Spasio-Temporal SAE per Tahun\nKecamatan Jawa Timur 2018-2025", fontsize=14, fontweight="bold")
plt.xlabel("Tahun", fontsize=12)
plt.ylabel("RRMSE STFH (%)", fontsize=12)
plt.grid(axis="y", linestyle="--", alpha=0.7)
plt.tight_layout()

# Menyimpan plot
output_plot_path = OUTPUT_DIR / "boxplot_rrmse_stfh_only.png"
plt.savefig(output_plot_path, dpi=300)
plt.show()

print(f"✓ Boxplot RRMSE STFH berhasil disimpan ke: {output_plot_path}")

✓ Boxplot RRMSE STFH berhasil disimpan ke: data/output_python_stfh/boxplot_rrmse_stfh_only.png


In [9]:
import numpy as np
import pandas as pd
from pathlib import Path
import json
import patsy
from scipy import sparse
from scipy.optimize import minimize
from scipy.sparse.linalg import splu

# 1. Mengembalikan definisi kelas STFH dari riwayat input secara aman
def restore_stfh_class_safely():
    # Cari kode sel pertama yang mendefinisikan class STFH
    for code in _ih:
        if "class STFH:" in code and "self.y_scale" in code:
            # Eksekusi di namespace global saat ini secara langsung agar fungsionalitas built-in utuh
            exec(code, globals())
            if "STFH" in globals():
                return globals()["STFH"]
    return None

STFH_Class = restore_stfh_class_safely()
if STFH_Class is not None:
    print("✓ Berhasil memulihkan kelas STFH dengan aman dari riwayat kernel.")
else:
    try:
        from __main__ import STFH as STFH_Class
        print("✓ Berhasil memulihkan kelas STFH dari __main__.")
    except ImportError:
        raise ValueError("Definisi kelas STFH tidak ditemukan. Silakan jalankan ulang sel pertama (c4Qu2EYKFXs6).")

# 2. Load Data Penunjang
OUTPUT_DIR = Path("data/output_python_stfh")
panel = read_panel()
shp = read_filter_shp(panel)
panel, D, T = align_panel(panel, shp)
W, _ = build_queen_knn(shp)

# Membaca hasil model penuh (baseline)
baseline_path = OUTPUT_DIR / "diagnostik_model.json"
with open(baseline_path, 'r') as f:
    baseline_meta = json.load(f)
baseline_aic = baseline_meta["AIC_approx"]
baseline_bic = baseline_meta["BIC_approx"]

# 3. Proses Seleksi Variabel (Backward Elimination)
print("\n=== PROSES SELEKSI VARIABEL (BACKWARD ELIMINATION) ===")
kandidat_variabel = ["CO", "LST", "NDVI", "NDBI", "NTL", "listrik", "pendidikan", "ekonomi"]
best_formula = "pengeluaran_mean ~ " + " + ".join(kandidat_variabel)
best_aic = baseline_aic

for r_step in range(len(kandidat_variabel)):
    scores = []
    for var in kandidat_variabel:
        test_vars = [v for v in kandidat_variabel if v != var]
        test_formula = f"pengeluaran_mean ~ {' + '.join(test_vars)}"
        try:
            model_test = STFH_Class(panel, W, D, T, MODEL, test_formula)
            fit_test = model_test.fit(maxiter=50, multistart=False)
            scores.append((fit_test["AIC_approx"], test_formula, var))
        except Exception as e:
            continue
    if not scores:
        break
    scores.sort(key=lambda x: x[0])
    min_aic, formula_opt, removed_var = scores[0]
    if min_aic < best_aic:
        print(f"Langkah {r_step+1}: Menghapus '{removed_var}' -> AIC turun dari {best_aic:.2f} menjadi {min_aic:.2f}")
        best_aic = min_aic
        best_formula = formula_opt
        kandidat_variabel.remove(removed_var)
    else:
        print("Tidak ada pengurangan variabel lebih lanjut yang menurunkan AIC.")
        break

print(f"\nFormula Terpilih Akhir: {best_formula}")

# 4. Fit Model Baru dengan Variabel Terseleksi
print("\n=== MENJALANKAN MODEL SELEKSI VARIABEL STFH ===")
fit_model_select = STFH_Class(panel, W, D, T, MODEL, best_formula)
fit_model_select.source_panel = panel.copy()
fitted_select = fit_model_select.fit()

# 5. Membandingkan Hasil Evaluasi Akurasi dan Sebaran RRMSE
print("\n=== EVALUASI DAN PERBANDINGAN AKURASI MODEL ===")
print(f"1. Kriteria Informasi:")
print(f"   - Baseline Model (Full)     -> AIC: {baseline_aic:.2f} | BIC: {baseline_bic:.2f}")
print(f"   - Selected Model (Optimized)-> AIC: {fitted_select['AIC_approx']:.2f} | BIC: {fitted_select['BIC_approx']:.2f}")

# Membaca file hasil akhir
hasil_path = OUTPUT_DIR / "hasil_pbmse_stfh_2018_2025.csv"
hasil_df = pd.read_csv(hasil_path)

eblup_full = hasil_df["EBLUP_STFH"].to_numpy()
eblup_select = fitted_select["fitted"]

rmse_full = hasil_df["RMSE_STFH"].to_numpy() if "RMSE_STFH" in hasil_df.columns else np.sqrt(hasil_df["MSE_STFH_pbmse"].to_numpy())
mse_select_approx = fitted_select["theta"]["sigma21"] + fitted_select["theta"]["sigma22"]
rmse_select = np.sqrt(mse_select_approx) * np.ones_like(eblup_select)

rrmse_full = 100 * rmse_full / np.abs(eblup_full)
rrmse_select = 100 * rmse_select / np.abs(eblup_select)

komparasi_df = pd.DataFrame({
    "Metrik": ["Rata-rata Estimasi", "Standar Deviasi Estimasi", "Rata-rata RRMSE (%)", "Median RRMSE (%)", "Maksimum RRMSE (%)"],
    "Model Penuh (Full)": [np.mean(eblup_full), np.std(eblup_full), np.mean(rrmse_full), np.median(rrmse_full), np.max(rrmse_full)],
    "Model Terseleksi (Selected)": [np.mean(eblup_select), np.std(eblup_select), np.mean(rrmse_select), np.median(rrmse_select), np.max(rrmse_select)]
})
print(komparasi_df.to_string(index=False))

[1] Membaca dan memvalidasi data panel
Kolom tanpa nama/indeks CSV dihapus: ['Unnamed: 0']
[2] Membaca dan memfilter SHP ke area panel
[3] Melengkapi dan mengurutkan panel sesuai SHP
D = 661 kecamatan, T = 8 tahun, N = 5288 baris
[4] Membuat matriks W Queen + KNN hanya untuk island
Island Queen sebelum KNN: 5
Island sesudah KNN: 0
[5] Estimasi REML model STFH
REML initial point 1/3: rho1=0.25, rho2=0.30
  success=True, iterations=16, REML_nll=4143.676405: CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH
REML initial point 2/3: rho1=0.00, rho2=0.00
  success=True, iterations=17, REML_nll=4143.676405: CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH
REML initial point 3/3: rho1=0.50, rho2=0.50
  success=True, iterations=18, REML_nll=4143.676405: CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH
Konvergensi: True | CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH
Estimasi parameter: {'sigma21': 7709580205.509383, 'rho1': 0.9407152768244473, 'sigma22': 45956678969.30627, 'rh

In [10]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

komparasi_df.to_csv(
    OUTPUT_DIR / "perbandingan_model_full_selected.csv",
    index=False
)

print("Hasil perbandingan model berhasil disimpan.")
pd.DataFrame({
    "EBLUP_Selected": np.asarray(eblup_select).ravel()
}).to_csv(
    OUTPUT_DIR / "hasil_estimasi_stfh_selected.csv",
    index=False
)

selected_meta = {
    "formula": best_formula,
    "jumlah_variabel": len(kandidat_variabel),
    "variabel_terpilih": kandidat_variabel,
    "variabel_dihapus": ["listrik"],
    "AIC_approx": float(fitted_select["AIC_approx"]),
    "BIC_approx": float(fitted_select["BIC_approx"]),
    "theta": {
        k: float(v)
        for k, v in fitted_select["theta"].items()
    },
    "catatan": (
        "AIC/BIC perlu diverifikasi untuk "
        "perbandingan fixed effects. "
        "PBMSE model selected belum dihitung."
    )
}

with open(
    OUTPUT_DIR / "diagnostik_model_selected.json",
    "w"
) as f:
    json.dump(selected_meta, f, indent=4)

Hasil perbandingan model berhasil disimpan.


In [11]:
import numpy as np
import pandas as pd
from pathlib import Path
import json
import sys

OUTPUT_DIR = Path("data/output_python_stfh")

# Memulihkan fungsi bootstrap dengan modifikasi formula fleksibel
def bootstrap_pbmse_selected(panel, W, model_type, theta, beta, formula_select, B, max_attempts, maxiter, seed, output, verbose=False):
    D = W.shape[0]
    T = panel[TIME_KEY].nunique()
    # Gunakan formula kustom model terseleksi, bukan formula global
    m0 = STFH_Class(panel, W, D, T, model_type, formula_select)
    mu = np.zeros(len(panel), dtype=float)
    for j, name in enumerate(m0.names):
        if name not in beta:
            raise ValueError(f"Koefisien {name} tidak ada pada hasil fit awal")
        mu += m0.X_raw[:, j] * beta[name]

    vardir = panel[VARDIR_KEY].to_numpy(dtype=float)
    rng = np.random.default_rng(seed)
    L = (sparse.eye(D, format="csc") - theta["rho1"] * W.tocsc()).tocsc()
    Llu = splu(L)
    s1, s2 = float(theta["sigma21"]), float(theta["sigma22"])
    r2 = float(theta["rho2"]) if model_type == "ST" else 0.0

    start1 = 0.5 * float(np.median(vardir))
    start2 = 0.5 * float(np.median(vardir))
    rho1_start, rho2_start = 0.5, 0.5

    sumsquares = np.zeros(D * T, dtype=float)
    accepted = 0
    attempts = 0
    log_rows = []

    while accepted < B and attempts < max_attempts:
        attempts += 1
        z = rng.standard_normal(D)
        u1 = Llu.solve(np.sqrt(s1) * z)
        u1dt = np.repeat(u1, T)

        innovations = rng.standard_normal((D, T)) * np.sqrt(s2)
        v = np.zeros((D, T), dtype=float)
        if model_type == "ST":
            v[:, 0] = innovations[:, 0] / np.sqrt(1.0 - r2**2)
            for t in range(1, T):
                v[:, t] = r2 * v[:, t - 1] + innovations[:, t]
        else:
            v = innovations
        u2dt = v.reshape(-1)

        e = rng.standard_normal(D * T) * np.sqrt(vardir)
        latent_truth = mu + u1dt + u2dt
        y_star = latent_truth + e
        simulated = panel.copy()
        simulated["pengeluaran_mean"] = y_star

        try:
            rep_model = STFH_Class(simulated, W, D, T, model_type, formula_select)
            # start values
            sigma1_scaled = start1 / rep_model.y_scale**2
            sigma2_scaled = start2 / rep_model.y_scale**2
            x0 = [np.log(sigma1_scaled), rho1_start, np.log(sigma2_scaled)]
            bounds = [(-25.0, 25.0), (-0.98, 0.98), (-25.0, 25.0)]
            if model_type == "ST":
                x0.append(rho2_start)
                bounds.append((-0.98, 0.98))

            opt = minimize(
                rep_model._evaluate, x0=np.asarray(x0, dtype=float), method="L-BFGS-B",
                bounds=bounds, options={"maxiter": int(maxiter), "ftol": 1e-9, "gtol": 1e-5}
            )
            if opt.success and np.isfinite(opt.fun):
                fitted_b = rep_model._evaluate(opt.x, details=True)
                differences = np.asarray(fitted_b["fitted"]) - latent_truth
                sumsquares += differences**2
                accepted += 1
        except Exception:
            continue

    if accepted < B:
        raise RuntimeError(f"Hanya {accepted} replikasi yang sukses dari {B} percobaan.")

    return sumsquares / B, {"B_sukses": accepted, "jumlah_percobaan": attempts}

# 1. Menjalankan Bootstrap Parametrik (PBMSE) untuk Model Terseleksi
print("=== MENJALANKAN PARAMETRIC BOOTSTRAP UNTUK MODEL TERSELEKSI (B=10) ===")
theta_select = fitted_select["theta"]
beta_select = fitted_select["beta"]

mse_select, diag_select = bootstrap_pbmse_selected(
    panel=panel,
    W=W,
    model_type=MODEL,
    theta=theta_select,
    beta=beta_select,
    formula_select=best_formula,
    B=10,
    max_attempts=50,
    maxiter=100,
    seed=2026,
    output=OUTPUT_DIR
)

# 2. Menghitung MSE, RMSE, RRMSE, dan CV untuk Model Terseleksi
hasil_select_df = panel.copy()
hasil_select_df["EBLUP_STFH_selected"] = fitted_select["fitted"]
hasil_select_df["MSE_STFH_selected"] = mse_select
hasil_select_df["RMSE_STFH_selected"] = np.sqrt(mse_select)

with np.errstate(divide="ignore", invalid="ignore"):
    hasil_select_df["RRMSE_STFH_selected"] = np.where(
        hasil_select_df["EBLUP_STFH_selected"] != 0,
        100.0 * hasil_select_df["RMSE_STFH_selected"] / np.abs(hasil_select_df["EBLUP_STFH_selected"]),
        np.nan
    )
    hasil_select_df["CV_STFH_selected"] = hasil_select_df["RRMSE_STFH_selected"]

# Simpan hasil rinci model terseleksi
hasil_select_df.to_csv(OUTPUT_DIR / "hasil_pbmse_stfh_selected.csv", index=False)
print("✓ Berhasil menyimpan hasil detail PBMSE model terseleksi ke 'hasil_pbmse_stfh_selected.csv'.\n")

# 3. Membaca Ulang Hasil Model Penuh (Full) untuk Perbandingan Konsisten
hasil_full_df = pd.read_csv(OUTPUT_DIR / "hasil_pbmse_stfh_2018_2025.csv")

# 4. Ringkasan Komparasi Rata-rata Tiga Model (Direct vs Full vs Selected)
komparasi_tiga = pd.DataFrame({
    "Tahun": TAHUN,
    "Rata_MSE_Direct": [hasil_full_df.loc[hasil_full_df["tahun"] == y, "VAR...8"].mean() for y in TAHUN],
    "Rata_MSE_Full": [hasil_full_df.loc[hasil_full_df["tahun"] == y, "MSE_STFH_pbmse"].mean() for y in TAHUN],
    "Rata_MSE_Selected": [hasil_select_df.loc[hasil_select_df["tahun"] == y, "MSE_STFH_selected"].mean() for y in TAHUN],

    "Rata_RMSE_Direct": [np.sqrt(hasil_full_df.loc[hasil_full_df["tahun"] == y, "VAR...8"]).mean() for y in TAHUN],
    "Rata_RMSE_Full": [hasil_full_df.loc[hasil_full_df["tahun"] == y, "RMSE_STFH"].mean() for y in TAHUN],
    "Rata_RMSE_Selected": [hasil_select_df.loc[hasil_select_df["tahun"] == y, "RMSE_STFH_selected"].mean() for y in TAHUN],

    "Rata_RRMSE_Direct": [hasil_full_df.loc[hasil_full_df["tahun"] == y, "RRMSE_direct"].mean() for y in TAHUN],
    "Rata_RRMSE_Full": [hasil_full_df.loc[hasil_full_df["tahun"] == y, "RRMSE_STFH"].mean() for y in TAHUN],
    "Rata_RRMSE_Selected": [hasil_select_df.loc[hasil_select_df["tahun"] == y, "RRMSE_STFH_selected"].mean() for y in TAHUN]
})

komparasi_tiga.to_csv(OUTPUT_DIR / "ringkasan_komparasi_tiga_model.csv", index=False)

print("=== RINGKASAN KOMPARASI RATA-RATA AKURASI TIGA MODEL ===")
display(komparasi_tiga)

# 5. Perbandingan Parameter Estetik
print("\n=== PARAMETER BOOTSTRAP DIAGNOSTICS ===")
print(f"Model STFH Awal (Full)     -> B Sukses: 10 | Percobaan: 10")
print(f"Model STFH Terseleksi (Selected)-> B Sukses: {diag_select['B_sukses']} | Percobaan: {diag_select['jumlah_percobaan']}")

=== MENJALANKAN PARAMETRIC BOOTSTRAP UNTUK MODEL TERSELEKSI (B=10) ===
✓ Berhasil menyimpan hasil detail PBMSE model terseleksi ke 'hasil_pbmse_stfh_selected.csv'.

=== RINGKASAN KOMPARASI RATA-RATA AKURASI TIGA MODEL ===


,Tahun,Rata_MSE_Direct,Rata_MSE_Full,Rata_MSE_Selected,Rata_RMSE_Direct,Rata_RMSE_Full,Rata_RMSE_Selected,Rata_RRMSE_Direct,Rata_RRMSE_Full,Rata_RRMSE_Selected
0,2018,2.354532e+10,8.721647e+09,8.711449e+09,103024.570470,77908.851999,77893.734131,10.527358,8.593841,8.594212
1,2019,5.172260e+10,7.720021e+09,7.721239e+09,107357.092955,73944.460000,73944.867558,10.330659,8.106422,8.106025
2,2020,2.391375e+10,8.939291e+09,8.933939e+09,110968.014194,80902.460257,80881.206564,10.680746,8.426755,8.424426
3,2021,2.054838e+10,8.626183e+09,8.627936e+09,104875.119813,78950.334181,78952.860075,8.598814,7.155576,7.156043
4,2022,3.734067e+10,1.160406e+10,1.160109e+10,137658.428958,93233.220490,93223.326240,11.611951,8.546156,8.546015
5,2023,7.904740e+10,1.378481e+10,1.377688e+10,169935.022400,102346.189222,102325.270612,12.123411,8.809140,8.807726
6,2024,5.181455e+10,1.356588e+10,1.356067e+10,160303.657454,101571.273235,101552.633163,11.473181,8.383550,8.383134
7,2025,6.201581e+10,1.390162e+10,1.388300e+10,153256.245541,100778.637730,100727.533887,10.645408,8.098661,8.094464



=== PARAMETER BOOTSTRAP DIAGNOSTICS ===
Model STFH Awal (Full)     -> B Sukses: 10 | Percobaan: 10
Model STFH Terseleksi (Selected)-> B Sukses: 10 | Percobaan: 10


In [12]:
import pandas as pd
from pathlib import Path

# 1. Load data hasil estimasi model terpilih (selected)
OUTPUT_DIR = Path("data/output_python_stfh")
filepath = OUTPUT_DIR / "hasil_pbmse_stfh_selected.csv"

if not filepath.exists():
    raise FileNotFoundError(f"File {filepath} tidak ditemukan. Sila jalankan sel sebelumnya terlebih dahulu.")

hasil_select = pd.read_csv(filepath)

# 2. Ambil kolom penting dan urutkan berdasarkan tahun dan nilai estimasi pengeluaran tertinggi
kolom_penting = ["tahun", "kode_kecamatan_kemendagri", "kode_kecamatan_bps", "pengeluaran_mean", "EBLUP_STFH_selected"]
ranking_df = hasil_select[kolom_penting].copy()

# Urutkan secara descending berdasarkan tahun dan nilai estimasi model terpilih
ranking_df = ranking_df.sort_values(by=["tahun", "EBLUP_STFH_selected"], ascending=[True, False])

# 3. Ambil top 3 kecamatan untuk setiap tahun menggunakan groupby + head
top_3_per_tahun = ranking_df.groupby("tahun").head(3).copy()

# Tambahkan kolom rank untuk memperjelas visualisasi
top_3_per_tahun["rank"] = top_3_per_tahun.groupby("tahun").cumcount() + 1

# Atur ulang susunan kolom agar rapi
top_3_per_tahun = top_3_per_tahun[["tahun", "rank", "kode_kecamatan_kemendagri", "kode_kecamatan_bps", "pengeluaran_mean", "EBLUP_STFH_selected"]]

# 5. Tampilkan tabel hasil perankingan
print("=== TOP 3 KECAMATAN DENGAN ESTIMASI PENGELUARAN TERTINGGI PER TAHUN (SELECTED MODEL) ===")
display(top_3_per_tahun.to_string(index=False))

=== TOP 3 KECAMATAN DENGAN ESTIMASI PENGELUARAN TERTINGGI PER TAHUN (SELECTED MODEL) ===


' tahun  rank kode_kecamatan_kemendagri  kode_kecamatan_bps  pengeluaran_mean  EBLUP_STFH_selected\n  2018     1                  35.07.22             3507300       3760129.017         3.760129e+06\n  2018     2                  35.78.08             3578100       2479982.071         2.384688e+06\n  2018     3                  35.78.03             3578070       2455804.166         2.305944e+06\n  2019     1                  35.07.22             3507300       3447245.690         2.901297e+06\n  2019     2                  35.15.05             3515050       2441388.701         2.441389e+06\n  2019     3                  35.78.08             3578100       2316760.846         2.396017e+06\n  2020     1                  35.13.02             3513020       3400520.858         3.400521e+06\n  2020     2                  35.78.23             3578020       3022455.941         3.022456e+06\n  2020     3                  35.78.08             3578100       2772191.228         2.483081e+06\n  2021   

In [13]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# 1. Load data hasil model terpilih
OUTPUT_DIR = Path("data/output_python_stfh")
filepath = OUTPUT_DIR / "hasil_pbmse_stfh_selected.csv"

if not filepath.exists():
    raise FileNotFoundError(f"File {filepath} tidak ditemukan. Silakan jalankan simulasi bootstrap model terpilih terlebih dahulu.")

hasil_select = pd.read_csv(filepath)

# Memastikan kolom RRMSE_STFH_selected ada (RRMSE setara dengan RSE dalam persen)
if "RRMSE_STFH_selected" not in hasil_select.columns:
    # Hitung jika belum ada
    hasil_select["RRMSE_STFH_selected"] = np.where(
        hasil_select["EBLUP_STFH_selected"] != 0,
        100.0 * hasil_select["RMSE_STFH_selected"] / np.abs(hasil_select["EBLUP_STFH_selected"]),
        np.nan
    )

# 2. Setup Plot
plt.figure(figsize=(10, 5))
sns.set_theme(style="whitegrid")

# Membuat boxplot horizontal
ax = sns.boxplot(
    data=hasil_select,
    x="RRMSE_STFH_selected",
    color="#2bc0e4",
    orient="h",
    width=0.4,
    flierprops={'marker': '.', 'markersize': 4, 'alpha': 0.5, 'markerfacecolor': '#e74c3c', 'markeredgecolor': '#e74c3c'}
)

# Menambahkan judul dan label
plt.title("Distribusi RSE Spasio-Temporal SAE", fontsize=14, fontweight="bold", pad=15)
plt.xlabel("RSE (%)", fontsize=12, labelpad=10)
plt.ylabel("Model Terseleksi", fontsize=12)

# Atur tampilan agar lebih rapi
plt.tight_layout()

# 3. Simpan Gambar
plot_path = OUTPUT_DIR / "boxplot_horizontal_rse_selected.png"
plt.savefig(plot_path, dpi=300, bbox_inches="tight")
plt.show()

print(f"✓ Boxplot horizontal RSE berhasil dibuat dan disimpan ke: {plot_path}")

✓ Boxplot horizontal RSE berhasil dibuat dan disimpan ke: data/output_python_stfh/boxplot_horizontal_rse_selected.png


In [16]:
import pandas as pd
from pathlib import Path

# 1. Muat data hasil estimasi selected model
OUTPUT_DIR = Path("data/output_python_stfh")
filepath = OUTPUT_DIR / "hasil_pbmse_stfh_selected.csv"

if not filepath.exists():
    raise FileNotFoundError(f"File {filepath} tidak ditemukan. Silakan jalankan simulasi bootstrap model terpilih terlebih dahulu.")

hasil_select = pd.read_csv(filepath)

# 2. Hitung persentase RRMSE < 25% per tahun menggunakan lambda untuk menghindari isu kompatibilitas pandas
ringkasan_persen_tahunan = hasil_select.groupby("tahun").apply(lambda x: pd.Series({
    "Total_Kecamatan": len(x),
    "RRMSE_di_bawah_25": (x["RRMSE_STFH_selected"] < 25.0).sum(),
    "Persentase_%": round(((x["RRMSE_STFH_selected"] < 25.0).sum() / len(x)) * 100, 2)
})).reset_index()

print("=== PROPORSI ESTIMASI SELECTED DENGAN RRMSE < 25% PER TAHUN ===")
display(ringkasan_persen_tahunan)

=== PROPORSI ESTIMASI SELECTED DENGAN RRMSE < 25% PER TAHUN ===


/tmp/ipykernel_675/3081501951.py:14: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  ringkasan_persen_tahunan = hasil_select.groupby("tahun").apply(lambda x: pd.Series({


,tahun,Total_Kecamatan,RRMSE_di_bawah_25,Persentase_%
0,2018,661.0,659.0,99.70
1,2019,661.0,659.0,99.70
2,2020,661.0,660.0,99.85
3,2021,661.0,661.0,100.00
4,2022,661.0,660.0,99.85
5,2023,661.0,661.0,100.00
6,2024,661.0,661.0,100.00
7,2025,661.0,660.0,99.85
